In [2]:
pip install sentence-transformers faiss-cpu langchain_community langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.1
    Uninstalling langchain-core-1.6.1:
      Successfully uninstalled langchain-core-1.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are install

In [3]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from langchain_openai import ChatOpenAI
from google.colab import userdata
import os
import requests
import json

#Extract content from a dynamic knowledge source through web crawling

I’ll integrate a dynamic knowledge base into your RAG system.
To do so, I’ll first gather content using the web crawling technique. Then, I’ll chunk it, create embeddings for the chunked content, and store the chunks in a Facebook AI Similarity Search (FAISS) database

Begin gathering the content from a website using web crawling technique

In [4]:
import requests
from bs4 import BeautifulSoup
import re
url = "https://thealliance.ai/"
try:
  resp = requests.get(url, timeout=15)
  resp.raise_for_status()
  soup = BeautifulSoup(resp.text, "html.parser")
  # Remove script/style/noscript tags then get visible text
  for s in soup(["script", "style", "noscript"]):
    s.decompose()
  global page_text
  page_text = soup.get_text(separator="\n")
  page_text = re.sub(r'\n\s*\n+', '\n\n', page_text) # collapse multiple blank lines
  # Optionally save to file
  with open("thealliance_ai_content.txt", "w", encoding="utf-8") as f:
    f.write(page_text)
  print(f"Crawled {url} -> saved to 'thealliance_ai_content.txt' (chars: {len(page_text)})")
except Exception as e:
  print("Web crawl failed. You can create `page_text` manually. Error:", e)

Crawled https://thealliance.ai/ -> saved to 'thealliance_ai_content.txt' (chars: 6608)


Chunk the content in thealliance_ai_content.txt

In [5]:
# Create chunks for web crawl
def split_into_chunks(text, chunk_size=300, overlap=50):
  chunks = []
  start = 0
  while start < len(text):
    end = start + chunk_size
    chunks.append(text[start:end])
    start += chunk_size - overlap
  return chunks

# Always run if page_text exists
if 'page_text' in globals():
  chunks_web = split_into_chunks(page_text)
  print(f"Web-crawled content split into {len(chunks_web)} chunks.")
else:
 raise RuntimeError("No web-crawled text found. Run the crawl first.")

Web-crawled content split into 27 chunks.


Convert the content chunks into vector embedding

In [6]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
# Ensure embedding model (will reuse if already present)
try:
  embed_model
except NameError:
  embed_model = SentenceTransformer('all-MiniLM-L6-v2')
  print("Loaded embedding model 'all-MiniLM-L6-v2'")
def build_index_from_chunks(chunks_local):
  embs = embed_model.encode(chunks_local, convert_to_numpy=True).astype(np.float32)
  dim = embs.shape[1]
  idx = faiss.IndexFlatL2(dim)
  idx.add(embs)
  return idx, embs

# Build web index if needed
if 'chunks_web' in globals():
  if 'index_web' not in globals():
    index_web, embeddings_web = build_index_from_chunks(chunks_web)
    chunk_store_web = chunks_web
    print(f"Built FAISS index for web source ({len(chunks_web)} chunks).")
  else:
    print("Web FAISS index already present.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded embedding model 'all-MiniLM-L6-v2'
Built FAISS index for web source (27 chunks).


#Test the system to assess the response quality
For this, I’ll enter a query, just like
an end user would, and assess the output the RAG system generates

In [7]:
user_query = "Tell more about AI Alliance"

Retrieve the top three responses to test the system. Retrieving only the top few results is a best practice that helps keep the context focused, reduces noise, and allows the model to generate more accurate and concise answers

In [8]:
import numpy as np
def retrieve_top_k(index_obj, chunk_store_local, query, k=3):
  q_emb = embed_model.encode([query], convert_to_numpy=True).astype(np.float32)
  distances, ids = index_obj.search(q_emb, k)
  top_chunks = [chunk_store_local[i] for i in ids[0]]
  return top_chunks, distances[0], ids[0]

retrieval_results = {}
if 'index_web' in globals() and 'chunk_store_web' in globals():
  top_chunks_web, dist_web, ids_web = retrieve_top_k(index_web, chunk_store_web, user_query, k=3)
  retrieval_results['web'] = {
    'top_chunks': top_chunks_web,
    'distances': dist_web,
    'ids': ids_web
  }
  print(f"Web: retrieved {len(top_chunks_web)} chunks.")
if not retrieval_results:
 raise RuntimeError("No index found. Run Step 7 to build indexes from chunks.")

Web: retrieved 3 chunks.


Review the retrieved chunks to verify whether the RAG system is extracting the latest information in response to the user query

In [9]:
for source in ['web']:
  if source in retrieval_results:
    rag_prompt = """Use the retreived web content to answer the question:"""
    rag_prompt += ("\n" + "="*8 + f" Retrieved chunks — {source.upper()} " + "="*8 + "\n")
    for i, ch in enumerate(retrieval_results[source]['top_chunks'], start=1):
      rag_prompt += (f"[{i}] {ch[:500].replace('\\n',' ')}\n")

rag_prompt += "\n" + "="*8
print(rag_prompt)


Use the retreived web content to answer the question:
======== Retrieved chunks — WEB ========
[1] ation and collaboration across communities of creators, scientists, academics and business leaders. This is a pivotal moment in defining the future of AI. IBM is proud to partner with like-minded organizations through the AI Alliance to ensure this open ecosystem drives an innovative AI agenda under
[2] k on safety. The AI Alliance brings together researchers, developers and companies to share tools and knowledge that can help us all make progress whether models are shared openly or not. We’re looking forward to working with partners to advance the state-of-the-art in AI and help everyone build res
[3] 

AI Alliance

About

Leadership

AI Alliance Association

Governance

Members

Projects

Project Tapestry

Join

Blog

Donate

AI Alliance

A global network advancing open, safe, and responsible AI through open innovation, collaboration, and advocacy

Home

AI Alliance

To accelerate open 



> OpenRouter runtime test



In [15]:
api_token = userdata.get('openrouter_key')

if not api_token:
  raise ValueError("REPLICATE_api_token not found or is empty in Colab Secrets. Please ensure it's set correctly.")

rag_prompt += f"""

Question:
{user_query}
"""

# Calling
response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": f"Bearer {api_token}",
    },
  data=json.dumps({
    "model": "inclusionai/ling-3.0-flash-vl",
    "messages": [
      {
        "role": "user",
        "content": rag_prompt
      }
    ]
  })
)


if response.status_code == 200:
  response = response.json()["choices"][0]["message"]["content"]
  print(response)
else:
  print(f"Error: {response.status_code} - {response.text}")

## AI Alliance

The **AI Alliance** is a **global network** dedicated to advancing **open, safe, and responsible AI** through open innovation, collaboration, and advocacy. Here are the key aspects:

### Mission & Vision
The AI Alliance aims to accelerate open innovation in AI and help define the future of the field. It serves as a pivotal moment in shaping how AI develops and is deployed responsibly.

### Community & Collaboration
The Alliance brings together a diverse and broad community, including:
- **Researchers** and **developers**
- **Companies** and **business leaders**
- **Scientists** and **academics**
- **Creators** across various domains

Notably, **IBM** is a prominent partner, proudly collaborating with like-minded organizations to ensure that an open ecosystem drives an innovative AI agenda.

### Knowledge Sharing & Safety
The AI Alliance facilitates the sharing of **tools and knowledge** among its members. It focuses on building responsible AI practices, whether models a